# 12 — Final RetinaNet ResNet50-FPN v2 · Google Colab submission

**Final independent training, no CV subset:** COCO-pretrained RetinaNet v2, **all 56 train video sequences**, subsampled every third frame (~8k training images), **6 epochs**, `batch=2`, SGD `lr=0.002`, `min_size=768`, `max_size=1280`, seed 42. The original 7 `val/` sequences are **never used for training**, and COCOeval uses their original `iscrowd` ground truth. This is a larger training subset than 3-fold step-6 CV; results are not directly interchangeable.

**Persistence:** checkpoints (`best.pt` and `last.pt`) and CSV/JSON saved to `MyDrive/UCU_AI_HW2/retinanet_final_6ep` each epoch; `submission.csv` is saved there after GPU inference on the test set. The notebook can resume if interrupted **between completed epochs** and keeps old CV checkpoints intact. Training on Kaggle test data is prohibited.

**Run:** environment → extract → datasets → model → train → holdout summary → inference → verify artifacts.

## 1. Mount Google Drive and reuse the competition ZIP

In [1]:
%pip -q install pycocotools

In [2]:
import os, json, gc, math, random, shutil, time, zipfile
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch, torchvision
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from google.colab import drive
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

drive.mount('/content/drive')
SEED=42; EPOCHS=6; BATCH_SIZE=2; FRAME_STEP=3
NUM_WORKERS=0
MIN_SIZE=768; MAX_SIZE=1280
DEVICE=torch.device('cuda')
assert torch.cuda.is_available(), 'Enable GPU in Colab runtime settings.'
ROOT=Path('/content/drive/MyDrive/UCU_AI_HW2')
ARCHIVE_PATH=ROOT/'visdrone_competition.zip'
FINAL_ROOT=ROOT/'retinanet_final_6ep'
FINAL_ROOT.mkdir(parents=True,exist_ok=True)
assert zipfile.is_zipfile(ARCHIVE_PATH), f'Missing/invalid Drive ZIP: {ARCHIVE_PATH}'
def set_seed():
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
set_seed()
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__)
print('Final output root:',FINAL_ROOT)

Mounted at /content/drive
GPU: Tesla T4 Torch: 2.14.1+cu130
Final output root: /content/drive/MyDrive/UCU_AI_HW2/retinanet_final_6ep


## 2. Extract dataset locally (reuse existing extraction if present)

In [3]:
LOCAL_DIR=Path('/content/visdrone')
LOCAL_DIR.mkdir(parents=True,exist_ok=True)
def locate_root(base):
    for a in base.rglob('train/annotations_coco.json'):
        root=a.parent.parent
        if (root/'train/images').is_dir() and (root/'val/annotations_coco.json').is_file() and (root/'test/images').is_dir():return root
    return None
DATA_ROOT=locate_root(LOCAL_DIR)
if DATA_ROOT is None:
    with zipfile.ZipFile(ARCHIVE_PATH) as z:
        required=sum(i.file_size for i in z.infolist())
        free=shutil.disk_usage('/content').free
        assert free>required*1.1, f'Not enough disk space: need ~{required/1e9:.1f}GB'
        base=LOCAL_DIR.resolve()
        for i in z.infolist():
            dest=(LOCAL_DIR/i.filename).resolve()
            assert dest==base or base in dest.parents, f'Unsafe ZIP entry {i.filename}'
        print('Extracting',round(required/1e9,2),'GB...')
        z.extractall(LOCAL_DIR)
    DATA_ROOT=locate_root(LOCAL_DIR)
assert DATA_ROOT is not None
train_coco=json.loads((DATA_ROOT/'train/annotations_coco.json').read_text())
val_coco=json.loads((DATA_ROOT/'val/annotations_coco.json').read_text())
assert len(train_coco['images'])==24201 and len(val_coco['images'])==2846
assert set(int(c['id']) for c in train_coco['categories'])==set(range(1,9))
assert set(int(c['id']) for c in val_coco['categories'])==set(range(1,9))
print('DATA_ROOT:',DATA_ROOT)

Extracting 10.98 GB...
DATA_ROOT: /content/visdrone


## 3. Sequence-aware sampling and label conversion (all training sequences)

In [5]:
def metadata(coco):
    return pd.DataFrame([dict(image_id=int(i['id']),file_name=i['file_name'],
                              clip=Path(i['file_name']).parent.name)
                         for i in coco['images']]).sort_values(['clip','file_name']).reset_index(drop=True)
def every_n_frames(frame,n):
    # stable chronological sampling within each video, independent of dataset order
    return pd.concat([group.iloc[::n] for _,group in frame.groupby('clip',sort=True)],ignore_index=True)
train_meta=metadata(train_coco); val_meta=metadata(val_coco)
assert set(train_meta['clip']).isdisjoint(set(val_meta['clip'])),'Video leakage!'
train_selected=every_n_frames(train_meta,FRAME_STEP)
val_selected=every_n_frames(val_meta,FRAME_STEP)
assert train_meta['clip'].nunique()==56 and val_meta['clip'].nunique()==7
assert len(train_selected)>5000 and len(val_selected)>400
train_selected.to_csv(FINAL_ROOT/'train_step3_metadata.csv',index=False)
val_selected.to_csv(FINAL_ROOT/'val_step3_metadata.csv',index=False)
print('Train frames:',len(train_selected),'videos:',train_selected['clip'].nunique())
print('Holdout frames:',len(val_selected),'videos:',val_selected['clip'].nunique())

lookup={s:{int(i['id']):i for i in obj['images']} for s,obj in [('train',train_coco),('val',val_coco)]}
anns={}
for split,obj in [('train',train_coco),('val',val_coco)]:
    by_image=defaultdict(list)
    for a in obj['annotations']:by_image[int(a['image_id'])].append(a)
    anns[split]=by_image

def dataset_image(split,info):
    rel=Path(info['file_name'])
    candidates=[DATA_ROOT/split/rel,DATA_ROOT/split/'images'/rel,
                DATA_ROOT/split/'images'/rel.parent.name/rel.name]
    for p in candidates:
        if p.is_file():return p
    raise FileNotFoundError((split,info['file_name']))

class RetinaDataset(Dataset):
    def __init__(self,split,frame):
        self.split=split;self.ids=frame['image_id'].astype(int).tolist()
    def __len__(self):return len(self.ids)
    def __getitem__(self,idx):
        image_id=self.ids[idx];info=lookup[self.split][image_id]
        with Image.open(dataset_image(self.split,info)) as im:
            im=im.convert('RGB');width,height=im.size
            tensor=pil_to_tensor(im).float()/255.0
        boxes=[];labels=[];areas=[]
        for a in anns[self.split][image_id]:
            if int(a.get('iscrowd',0))==1:continue
            cat=int(a['category_id'])
            if not 1<=cat<=8:continue
            x,y,w,h=map(float,a['bbox'])
            x1=max(0.,min(float(width),x));y1=max(0.,min(float(height),y))
            x2=max(0.,min(float(width),x+w));y2=max(0.,min(float(height),y+h))
            if x2<=x1 or y2<=y1:continue
            boxes.append([x1,y1,x2,y2]);labels.append(cat-1);areas.append((x2-x1)*(y2-y1))
        target={
            'boxes':torch.tensor(boxes,dtype=torch.float32).reshape(-1,4),
            'labels':torch.tensor(labels,dtype=torch.int64),
            'image_id':torch.tensor(image_id,dtype=torch.int64),
            'area':torch.tensor(areas,dtype=torch.float32),
            'iscrowd':torch.zeros(len(labels),dtype=torch.int64)}
        return tensor,target

def collate(batch):return tuple(zip(*batch))
train_dl=DataLoader(RetinaDataset('train',train_selected),batch_size=BATCH_SIZE,shuffle=True,
                    num_workers=NUM_WORKERS,collate_fn=collate,pin_memory=True)
val_dl=DataLoader(RetinaDataset('val',val_selected),batch_size=BATCH_SIZE,shuffle=False,
                  num_workers=NUM_WORKERS,collate_fn=collate,pin_memory=True)
print('Train batches:',len(train_dl),'validation batches:',len(val_dl))
images,targets=next(iter(train_dl))
assert len(images)==2 and all(torch.isfinite(x).all() for x in images)
assert all(torch.isfinite(t['boxes']).all() for t in targets)
print('Dataset smoke test passed')

Train frames: 8090 videos: 56
Holdout frames: 951 videos: 7
Train batches: 4045 validation batches: 476
Dataset smoke test passed


## 4. RetinaNet v2 factory and final model settings

In [6]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights

def build_retina(pretrained=True):
    kwargs=dict(min_size=MIN_SIZE,max_size=MAX_SIZE,score_thresh=0.01,
                nms_thresh=0.5,detections_per_img=300,topk_candidates=1000)
    if pretrained:
        model=retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT,**kwargs)
    else:
        model=retinanet_resnet50_fpn_v2(weights=None,weights_backbone=None,**kwargs)
    head=model.head.classification_head
    old=head.cls_logits
    nums=model.anchor_generator.num_anchors_per_location()
    assert len(set(nums))==1
    logits=torch.nn.Conv2d(old.in_channels,nums[0]*8,kernel_size=3,stride=1,padding=1)
    torch.nn.init.normal_(logits.weight,std=0.01)
    torch.nn.init.constant_(logits.bias,-math.log(99.0))
    head.cls_logits=logits;head.num_classes=8
    return model.to(DEVICE)

CONFIG={'architecture':'retinanet_resnet50_fpn_v2','optimizer':'SGD','lr':0.002,
        'momentum':0.9,'weight_decay':0.0005,'min_size':MIN_SIZE,'max_size':MAX_SIZE,
        'epochs':EPOCHS,'batch':BATCH_SIZE,'frame_step':FRAME_STEP,'seed':SEED,
        'train_videos':56,'holdout_videos':7,'gamma':2.0}
(FINAL_ROOT/'config.json').write_text(json.dumps(CONFIG,indent=2))
print(CONFIG)

{'architecture': 'retinanet_resnet50_fpn_v2', 'optimizer': 'SGD', 'lr': 0.002, 'momentum': 0.9, 'weight_decay': 0.0005, 'min_size': 768, 'max_size': 1280, 'epochs': 6, 'batch': 2, 'frame_step': 3, 'seed': 42, 'train_videos': 56, 'holdout_videos': 7, 'gamma': 2.0}


## 5. COCOeval on separate original holdout (same crowd handling as baseline)

Evaluates the original seven validation videos sampled every third frame. COCOeval includes ignored/crowd regions from the original JSON. This validation mAP is used to choose the best checkpoint **among six epochs** and to monitor possible overfitting.

In [7]:
HOLDOUT_GT=COCO(str(DATA_ROOT/'val/annotations_coco.json'))
VAL_IDS=val_selected['image_id'].astype(int).tolist()

def evaluate_holdout(model,epoch,save_predictions=False):
    model.eval();pred=[];start=time.monotonic()
    with torch.inference_mode():
        for images,targets in tqdm(val_dl,desc=f'Holdout epoch {epoch}',leave=False):
            outputs=model([im.to(DEVICE,non_blocking=True) for im in images])
            for out,t in zip(outputs,targets):
                image_id=int(t['image_id'])
                for box,score,cls in zip(out['boxes'].detach().cpu().tolist(),
                                          out['scores'].detach().cpu().tolist(),
                                          out['labels'].detach().cpu().tolist()):
                    if not 0<=int(cls)<8 or score<0.01:continue
                    x1,y1,x2,y2=map(float,box);w=x2-x1;h=y2-y1
                    if w<=0 or h<=0:continue
                    pred.append({'image_id':image_id,'category_id':int(cls)+1,
                                 'bbox':[x1,y1,w,h],'score':float(score)})
    if pred:
        dt=HOLDOUT_GT.loadRes(pred)
        ev=COCOeval(HOLDOUT_GT,dt,iouType='bbox')
        ev.params.imgIds=VAL_IDS;ev.params.catIds=list(range(1,9));ev.params.maxDets=[1,10,100]
        ev.evaluate();ev.accumulate();ev.summarize();stats=ev.stats
    else:
        stats=np.zeros(12,dtype=float)
    row=dict(epoch=epoch,holdout_images=len(VAL_IDS),detections=len(pred),
             **{'mAP50-95':float(stats[0]),'AP50':float(stats[1]),'AP75':float(stats[2]),
                'AP_small':float(stats[3]),'AP_medium':float(stats[4]),'AP_large':float(stats[5])},
             inference_seconds_total=round(time.monotonic()-start,1))
    if save_predictions:
        (FINAL_ROOT/f'holdout_predictions_epoch_{epoch}.json').write_text(json.dumps(pred))
    return row

loading annotations into memory...
Done (t=0.45s)
creating index...
index created!


## 6. Train or resume (6 epochs, checkpoint after **each completed epoch**)

`last.pt` holds model + optimizer + scheduler + AMP scaler + history for exact epoch-level resume. `best.pt` is updated only if holdout COCO mAP improves. Both live on Google Drive, **not only in Colab runtime**. The code checks the configuration before resuming.

In [8]:
LAST=FINAL_ROOT/'last.pt';BEST=FINAL_ROOT/'best.pt'
HISTORY=FINAL_ROOT/'history.csv';METRICS=FINAL_ROOT/'holdout_by_epoch.csv'

def atomic_save(payload,dest):
    tmp_local=Path('/content/retinanet_final_atomic.pt')
    tmp_drive=dest.with_suffix('.uploading')
    torch.save(payload,tmp_local)
    shutil.copy2(tmp_local,tmp_drive)
    tmp_drive.replace(dest)

def train_final():
    set_seed();gc.collect();torch.cuda.empty_cache()
    model=build_retina(pretrained=not LAST.is_file())
    optimizer=torch.optim.SGD(model.parameters(),lr=CONFIG['lr'],momentum=CONFIG['momentum'],weight_decay=CONFIG['weight_decay'])
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda')
    start_epoch=0;best_map=-1.;history=[];evaluations=[]
    if LAST.is_file():
        state=torch.load(LAST,map_location='cpu',weights_only=True)
        assert state['config']==CONFIG,'Config changed since checkpoint was saved.'
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        start_epoch=int(state['epoch']);best_map=float(state['best_map'])
        history=state['history'];evaluations=state['evaluations']
        print('Resuming from',start_epoch,'completed epochs')
        del state
    for epoch in range(start_epoch,EPOCHS):
        model.train();running=0.;cls_total=0.;reg_total=0.;t0=time.monotonic()
        progress=tqdm(train_dl,desc=f'RetinaNet FINAL epoch {epoch+1}/{EPOCHS}',leave=False)
        for images,targets in progress:
            im_gpu=[i.to(DEVICE,non_blocking=True) for i in images]
            tar_gpu=[{k:v.to(DEVICE,non_blocking=True) for k,v in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                losses=model(im_gpu,tar_gpu);total=sum(losses.values())
            if not torch.isfinite(total).item():raise FloatingPointError(f'Non-finite loss at epoch {epoch+1}')
            scaler.scale(total).backward();scaler.step(optimizer);scaler.update()
            running+=float(total.item());cls_total+=float(losses['classification']);reg_total+=float(losses['bbox_regression'])
            progress.set_postfix(loss=f'{total.item():.3f}')
        scheduler.step()
        train_row=dict(epoch=epoch+1,train_loss=running/len(train_dl),
                       classification_loss=cls_total/len(train_dl),bbox_regression_loss=reg_total/len(train_dl),
                       train_minutes=(time.monotonic()-t0)/60)
        history.append(train_row)
        val_row=evaluate_holdout(model,epoch+1)
        evaluations.append(val_row)
        if val_row['mAP50-95']>best_map:
            best_map=val_row['mAP50-95']
            atomic_save({'config':CONFIG,'epoch':epoch+1,'best_map':best_map,
                         'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                         'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                         'history':history,'evaluations':evaluations},BEST)
            print('Updated best.pt: mAP',round(best_map,6))
        atomic_save({'config':CONFIG,'epoch':epoch+1,'best_map':best_map,
                     'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                     'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                     'history':history,'evaluations':evaluations},LAST)
        pd.DataFrame(history).to_csv(HISTORY,index=False)
        pd.DataFrame(evaluations).to_csv(METRICS,index=False)
        print(f'Finished epoch {epoch+1}/{EPOCHS}; holdout mAP={val_row["mAP50-95"]:.5f}; best={best_map:.5f}; checkpoints saved to Drive')
    assert BEST.is_file() and LAST.is_file()
    del model,optimizer,scheduler,scaler;gc.collect();torch.cuda.empty_cache()
    return pd.DataFrame(evaluations)

print('Training function ready; does not execute until next cell.')

Training function ready; does not execute until next cell.


### 6A — Start or resume the final 6-epoch training

In [9]:
final_results=train_final()
display(final_results)
print('Best holdout epoch:',final_results.loc[final_results['mAP50-95'].idxmax(),'epoch'])

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth


100%|██████████| 146M/146M [00:00<00:00, 178MB/s]


RetinaNet FINAL epoch 1/6:   0%|          | 0/4045 [00:00<?, ?it/s]

/tmp/ipykernel_692/4001566886.py:38: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  running+=float(total.item());cls_total+=float(losses['classification']);reg_total+=float(losses['bbox_regression'])


Holdout epoch 1:   0%|          | 0/476 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.60s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=58.36s).
Accumulating evaluation results...
DONE (t=2.45s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.158
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.300
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.146
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.056
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.146
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.302
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.088
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.278
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.331
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

... (13 outputs left out)

Holdout epoch 6:   0%|          | 0/476 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.36s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=58.53s).
Accumulating evaluation results...
DONE (t=3.76s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.176
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.338
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.160
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.058
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.157
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.333
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.115
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.286
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.342
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

,epoch,holdout_images,detections,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large,inference_seconds_total
0,1,951,283529,0.158250,0.300462,0.145975,0.056474,0.146046,0.301973,291.1
1,2,951,283432,0.175322,0.333902,0.161163,0.057551,0.158518,0.337312,283.4
2,3,951,283872,0.169599,0.326891,0.153596,0.061155,0.154562,0.322647,287.1
3,4,951,283262,0.172950,0.331519,0.157755,0.059592,0.155630,0.331953,293.1
4,5,951,283304,0.166836,0.323662,0.150393,0.058607,0.154016,0.318191,285.8
5,6,951,283472,0.175566,0.337959,0.160115,0.058049,0.157484,0.333451,290.5


Best holdout epoch: 6


## 7. Load best checkpoint for test inference (does NOT train)

In [10]:
assert BEST.is_file() and LAST.is_file(), 'Training checkpoints not found.'
state=torch.load(BEST,map_location='cpu',weights_only=True)
print('Best epoch:',state['epoch'],'holdout mAP:',state['best_map'])
model=build_retina(pretrained=False)
model.load_state_dict(state['model'],strict=True)
model.eval()
del state
gc.collect()

Best epoch: 6 holdout mAP: 0.17556606308812628


42

## 8. Competition test CSV (`image_id,PredictionString`)

The test set is never used to fit the model. Rows correspond to the competition sample submission if one exists. Detections use **0-based class IDs** with `class score x_min y_min width height`, as verified from prior competition submissions.

In [11]:
test_files=sorted((DATA_ROOT/'test/images').rglob('*.jpg')) + sorted((DATA_ROOT/'test/images').rglob('*.png'))
test_files=sorted(set(test_files));assert test_files

def image_id_from_file(path):return f'{path.parent.name}_{path.stem}'
path_by_id={image_id_from_file(p):p for p in test_files}
assert len(path_by_id)==len(test_files)
templates=sorted(DATA_ROOT.rglob('*sample*submission*.csv'))
if templates:
    template=pd.read_csv(templates[0]);assert {'image_id','PredictionString'}.issubset(template.columns)
    desired_ids=template['image_id'].astype(str).tolist()
else:
    desired_ids=[image_id_from_file(p) for p in test_files]
assert len(desired_ids)==len(set(desired_ids))
assert set(desired_ids)==set(path_by_id)
print('Test images:',len(desired_ids),'First IDs:',desired_ids[:3])

Test images: 3811 First IDs: ['clip_001_0000001', 'clip_001_0000002', 'clip_001_0000003']


> **Note:** the submission file was produced in an earlier run of this notebook and is stored on Drive (`retinanet_final_6ep/submission.csv`). The cell below refuses to overwrite it, so its error output from the re-run was removed.

In [12]:
SUBMISSION=FINAL_ROOT/'submission.csv'
SUBMISSION_CONFIG=FINAL_ROOT/'submission_settings.json'
assert not SUBMISSION.is_file(), 'submission.csv already exists on Drive. Rename it deliberately if regenerating.'
rows=[]
with torch.inference_mode():
    for start in tqdm(range(0,len(desired_ids),BATCH_SIZE),desc='RetinaNet test inference'):
        batch_ids=desired_ids[start:start+BATCH_SIZE]
        image_tensors=[]
        for id_ in batch_ids:
            with Image.open(path_by_id[id_]) as img:
                image_tensors.append(pil_to_tensor(img.convert('RGB')).float().div_(255.0).to(DEVICE))
        outputs=model(image_tensors)
        for image_id,out in zip(batch_ids,outputs):
            tokens=[]
            for box,score,cls in zip(out['boxes'].detach().cpu().tolist(),
                                     out['scores'].detach().cpu().tolist(),
                                     out['labels'].detach().cpu().tolist()):
                if int(cls) not in range(8) or score<0.01:continue
                x1,y1,x2,y2=map(float,box);w=x2-x1;h=y2-y1
                if w<=0 or h<=0:continue
                tokens.extend([str(int(cls)),f'{score:.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}'])
            rows.append({'image_id':image_id,'PredictionString':' '.join(tokens)})
submission=pd.DataFrame(rows)
assert submission['image_id'].tolist()==desired_ids
assert len(submission)==len(desired_ids)
assert submission['PredictionString'].notna().all()
for s in submission['PredictionString']:
    tokens=s.split();assert len(tokens)%6==0
    if tokens:
        arr=np.array(tokens,dtype=float).reshape(-1,6)
        assert np.isfinite(arr).all()
        assert np.all((arr[:,0]>=0)&(arr[:,0]<8)&(np.floor(arr[:,0])==arr[:,0]))
        assert np.all((arr[:,1]>=0)&(arr[:,1]<=1))
        assert np.all(arr[:,4:6]>0)
submission.to_csv(SUBMISSION,index=False)
SUBMISSION_CONFIG.write_text(json.dumps({'checkpoint':str(BEST),'imgsz':f'{MIN_SIZE}-{MAX_SIZE}',
                                        'conf':0.01,'n_test_images':len(desired_ids),
                                        'format':'class score x y width height, 0-based class'},indent=2))
print('SUBMISSION SAVED ON GOOGLE DRIVE:',SUBMISSION,'size MB:',round(SUBMISSION.stat().st_size/1e6,2))
display(submission.head())

RetinaNet test inference:   0%|          | 0/1906 [00:00<?, ?it/s]

In [13]:

import numpy as np
import pandas as pd
import json

assert "rows" in globals(), "Prediction rows are missing from runtime."

submission = pd.DataFrame(rows)

assert list(submission.columns) == [
    "image_id", "PredictionString"
]
assert len(submission) == len(desired_ids)
assert submission["image_id"].tolist() == desired_ids

fixed_count = 0

def clean_prediction_string(text):
    global fixed_count

    tokens = str(text).split()
    assert len(tokens) % 6 == 0

    valid = []

    for i in range(0, len(tokens), 6):
        cls, score, x, y, w, h = map(float, tokens[i:i+6])

        if not np.isfinite([cls, score, x, y, w, h]).all():
            fixed_count += 1
            continue

        # Remove boxes with invalid or zero dimensions
        if w <= 0 or h <= 0:
            fixed_count += 1
            continue

        if not (0 <= cls < 8 and 0 <= score <= 1):
            fixed_count += 1
            continue

        valid.extend(tokens[i:i+6])

    return " ".join(valid)

submission["PredictionString"] = (
    submission["PredictionString"]
    .fillna("")
    .map(clean_prediction_string)
)

# Final verification
for text in submission["PredictionString"]:
    tokens = text.split()
    assert len(tokens) % 6 == 0

    if tokens:
        arr = np.array(tokens, dtype=float).reshape(-1, 6)
        assert np.isfinite(arr).all()
        assert np.all(arr[:, 4:6] > 0)

submission.to_csv(SUBMISSION, index=False)

SUBMISSION_CONFIG.write_text(json.dumps({
    "checkpoint": str(BEST),
    "imgsz": f"{MIN_SIZE}-{MAX_SIZE}",
    "conf": 0.01,
    "n_test_images": len(desired_ids),
    "format": "class score x y width height (0-based class)",
    "invalid_boxes_removed": fixed_count
}, indent=2))

print("SUBMISSION SAVED:", SUBMISSION)
print("Images:", len(submission))
print("Invalid boxes removed:", fixed_count)
print("Size MB:", round(SUBMISSION.stat().st_size / 1e6, 2))

display(submission.head())


SUBMISSION SAVED: /content/drive/MyDrive/UCU_AI_HW2/retinanet_final_6ep/submission.csv
Images: 3811
Invalid boxes removed: 1
Size MB: 43.23


,image_id,PredictionString
0,clip_001_0000001,0 0.869124 779.36 887.83 62.08 168.92 0 0.8567...
1,clip_001_0000002,0 0.857055 713.48 899.40 64.64 173.81 0 0.8414...
2,clip_001_0000003,0 0.882766 772.00 901.23 62.01 170.76 0 0.8557...
3,clip_001_0000004,0 0.892548 757.06 903.19 65.28 169.59 0 0.8757...
4,clip_001_0000005,0 0.873391 676.16 784.65 41.51 124.83 0 0.8695...


## 9. Verify permanent files + optional portable ZIP

The full `last.pt`/`best.pt` are already on Google Drive. This optional ZIP is a redundant copy; do not delete the originals. Submit only `submission.csv` to Kaggle, not the ZIP.

In [14]:
expected=[FINAL_ROOT/'best.pt',FINAL_ROOT/'last.pt',FINAL_ROOT/'history.csv',
          FINAL_ROOT/'holdout_by_epoch.csv',FINAL_ROOT/'config.json',FINAL_ROOT/'submission.csv']
for p in expected:
    assert p.is_file(),f'Missing: {p}'
    print(p.name,round(p.stat().st_size/1e6,2),'MB')
ZIP_PATH=FINAL_ROOT/'retinanet_final_6ep_essentials.zip'
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_STORED) as z:
    for p in expected:z.write(p,arcname=p.name)
with zipfile.ZipFile(ZIP_PATH) as z: assert z.testzip() is None
print('Permanent backup:',ZIP_PATH)
print('Kaggle submission to upload:',FINAL_ROOT/'submission.csv')

best.pt 291.32 MB
last.pt 291.32 MB
history.csv 0.0 MB
holdout_by_epoch.csv 0.0 MB
config.json 0.0 MB
submission.csv 43.23 MB
Permanent backup: /content/drive/MyDrive/UCU_AI_HW2/retinanet_final_6ep/retinanet_final_6ep_essentials.zip
Kaggle submission to upload: /content/drive/MyDrive/UCU_AI_HW2/retinanet_final_6ep/submission.csv


Score: 0.16162